<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs pandas and polars, the two libraries timed against each other below on file reads, column selection, filters, group-by, sorts, and rolling averages, plus lxml so pandas can parse the S&P 500 table off Wikipedia.

In [ ]:
!pip install pandas polars lxml

OpenBB is left out of that command on purpose because it pins versions across a long dependency list and can downgrade packages you already depend on. Install it separately in a fresh virtual environment following the OpenBB documentation.

## Imports and setup

We use pandas to read the Wikipedia membership table and hold the price data, polars to hold the same data in its columnar layout, and the OpenBB SDK to download daily prices.

In [ ]:
import pandas as pd
import polars as pl
import yfinance as yf

## Load the price data once

We download daily prices for a sample tickers back to 1990 and hold the result in a pandas DataFrame.

In [ ]:
tickers = TICKERS = [
    "NVDA", "AAPL", "MSFT", "GOOGL", "AMZN",
    "META", "TSLA", "AVGO", "TSM", "ORCL",
    "CSCO", "AMAT", "LRCX", "BRK-B", "JPM",
    "BAC", "V", "MA", "JNJ", "LLY",
    "ABBV", "XOM", "CVX", "WMT", "COST",
]
df_pandas = yf.download(tickers, start="1990-01-01")

A pandas DataFrame is what most questions in the Getting Started With Python for Quant Finance community start from, and it's what Zipline Reloaded's run_algorithm returns while PyFolio expects a pandas Series of daily returns. The cell fires 500 separate requests, so run it once and write the result to disk rather than re-fetching after every kernel restart.

We copy the same table into Polars so both libraries hold identical data before we time anything.

In [ ]:
df_polars = pl.from_pandas(df_pandas)

Both frames now hold the same rows, the same columns, and the same dtypes, so the timings below differ only by library. pl.from_pandas converts each column through Arrow, and for the numeric price columns here that lands as one contiguous block, which is the layout that lets Polars split work across CPU cores.

## Time reads, filters, and grouping

First the file read. The %timeit magic runs the statement many times and reports the average, so one slow disk hit doesn't decide the result.

In [ ]:
%timeit pd.read_csv("data.csv")

In [ ]:
%timeit pl.scan_csv("data.csv")

458 milliseconds for the pandas read on a 3.8 million row local CSV, against 3.57 milliseconds for the Polars line, both off one machine with the file already sitting on disk. Most of that gap is deferred work, because pl.scan_csv doesn't read the file, it builds a plan and waits until you ask for results. Swap in pl.read_csv when you want a straight read against a read.

Cut the 500 columns down to the first 100 tickers and see what each library charges to hand back only those.

In [ ]:
selected = tickers[:100]

In [ ]:
%timeit df_pandas[selected]

In [ ]:
%timeit df_polars.select(pl.col(selected))

A 500 column frame gets narrowed to a sector or a watchlist every time you look at a subset, and the same call repeats all session. Polars names columns with pl.col inside an expression instead of indexing the frame directly, and that one line is what I get asked about most by people moving over from pandas.

Keeping only the days GE closed above 100 is the row filter that runs ahead of any conditional study.

In [ ]:
%timeit df_pandas[df_pandas["GE"] > 100]

In [ ]:
%timeit df_polars.filter(pl.col("GE") > 100)

pandas builds a full array of True and False values and then uses it to index the frame. Polars hands the comparison to filter as an expression it can plan before running. On a single column the two land close enough that the ordering flips between runs.

A group-by sorts rows into buckets that share a value and then averages each bucket, and here it does that across millions of rows.

In [ ]:
%timeit df_pandas.groupby("GE").mean()

In [ ]:
%timeit df_polars.groupby("GE").mean()

Grouping by a price column isn't something you'd do in real work, and it's here because it forces both libraries to sort and aggregate the full frame. In a normal workflow you'd group by ticker or by month to get per-symbol or per-period numbers. Polars runs the aggregation across threads by default, so the size of the gap here depends on how many cores the machine has.

## Time column math and windows

Daily percent return for GE, attached as a new column, is the first thing I calculate on any price series.

In [ ]:
%timeit df_pandas.assign(GE_Return=df_pandas["GE"].pct_change())

In [ ]:
%timeit df_polars.with_columns((pl.col("GE").pct_change()).alias("GE_return"))

Percent change from one row to the next is what feeds the cumulative return curve and the annualized volatility number. pandas uses assign, Polars uses with_columns and alias to name the output, and both hand back a new frame rather than editing the original.

GE has no row on holidays and no row before it traded, and this fills those missing values with zero before any math runs.

In [ ]:
%timeit df_pandas["GE"].fillna(0)

In [ ]:
%timeit df_polars.with_columns(pl.col("GE").fill_null(0))

A zero close would read as a 100% loss the next time pct_change ran, so for prices I carry the last observed value forward and keep the zero fill for share volume, where no trades really does mean zero. Forward fill is also the one operation where Polars came out behind pandas in my notebook, and I never worked out why.

Sorting the whole frame by the GE column touches every row, which makes it the bluntest test here.

In [ ]:
%timeit df_pandas.sort_values("GE")

In [ ]:
%timeit df_polars.sort("GE")

A sort is a fair stress test of how each library shuffles data in memory. My own sorts are by date after stitching files together, or by trailing return before taking the top names, and it's an easy call to repeat by accident inside a loop.

A 20-day rolling average of GE recalculates the mean of the last 20 closes at every row.

In [ ]:
%timeit df_pandas.GE.rolling(window=20).mean()

In [ ]:
%timeit df_polars.with_columns(pl.col("GE").rolling_mean(20))

This is the calculation the daily job ran across every contract when I was building analytics for a metals trader, and it's the basis of most trend rules. Both versions look only backward, so today's average never includes tomorrow's price. If a rolling window is what's slowing your notebook, %timeit on both lines tells you whether the rewrite is worth doing.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.